# Sesión 2 · Tipos de aprendizaje y preparación de datos de seguridad
**IA aplicada a la ciberseguridad · Máster en Ciberseguridad**

Laboratorio en tres frentes: **NSL-KDD** (flujos), un **auth.log sintético** (logs) y un **pipeline de scikit-learn** sin fuga de datos.

| Parte | Qué haces |
|---|---|
| A | Perfilar y limpiar NSL-KDD |
| B | De 23 tipos de ataque a 4 categorías |
| C | Codificar variables categóricas |
| D | Escalar con lo que sabe el entrenamiento |
| E | Generar, parsear y convertir un auth.log en características |
| F | Tres divisiones y un pipeline sin fuga |
| G | Ampliación autónoma: desbalanceo extremo con SMOTE |

> **Cómo usar este cuaderno**
> - **Google Colab:** *Archivo › Subir cuaderno*, ejecuta la celda de preparación y sigue en orden. Los ficheros de la sesión (data/, utils.py) se borran al desconectar: vuelve a ejecutar la preparación al retomar.
> - **VS Code o GitHub Codespaces:** abre la carpeta del proyecto, elige el kernel de Python (*Select Kernel*) y ejecuta la celda de preparación.
> - Ejecuta las celdas **en orden** con `Mayús + Intro`. Antes de entregar: reinicia el kernel y ejecuta todo.

## Preparación del entorno
Descarga NSL-KDD si falta y crea `utils.py` si no lo traes de la sesión 1.

In [ ]:
# Preparación del entorno: ejecuta esta celda la primera (Colab, VS Code o Codespaces)
import sys, os, subprocess, importlib.util, urllib.request

EN_COLAB = "google.colab" in sys.modules
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter en VS Code / Codespaces")
print("Python:", sys.version.split()[0])

UTILS_BASE = "import os, urllib.request\nimport pandas as pd\n\nNSL_URL = \"https://raw.githubusercontent.com/defcom17/NSL_KDD/master/\"\n\nNSL_COLS = [\"duration\",\"protocol_type\",\"service\",\"flag\",\"src_bytes\",\"dst_bytes\",\"land\",\n  \"wrong_fragment\",\"urgent\",\"hot\",\"num_failed_logins\",\"logged_in\",\"num_compromised\",\n  \"root_shell\",\"su_attempted\",\"num_root\",\"num_file_creations\",\"num_shells\",\n  \"num_access_files\",\"num_outbound_cmds\",\"is_host_login\",\"is_guest_login\",\"count\",\n  \"srv_count\",\"serror_rate\",\"srv_serror_rate\",\"rerror_rate\",\"srv_rerror_rate\",\n  \"same_srv_rate\",\"diff_srv_rate\",\"srv_diff_host_rate\",\"dst_host_count\",\n  \"dst_host_srv_count\",\"dst_host_same_srv_rate\",\"dst_host_diff_srv_rate\",\n  \"dst_host_same_src_port_rate\",\"dst_host_srv_diff_host_rate\",\"dst_host_serror_rate\",\n  \"dst_host_srv_serror_rate\",\"dst_host_rerror_rate\",\"dst_host_srv_rerror_rate\",\n  \"label\",\"difficulty\"]\n\n\ndef load_nsl_kdd(split=\"train\", base=\"data\"):\n    \"\"\"Carga NSL-KDD y añade la columna is_attack.\"\"\"\n    name = (\"KDDTrain+.txt\" if split == \"train\"\n            else \"KDDTest+.txt\")\n    path = os.path.join(base, name)\n    if not os.path.exists(path):   # descarga si falta\n        os.makedirs(base, exist_ok=True)\n        url = NSL_URL + name.replace(\"+\", \"%2B\")\n        urllib.request.urlretrieve(url, path)\n    df = pd.read_csv(path, header=None, names=NSL_COLS)\n    df[\"is_attack\"] = (df[\"label\"] != \"normal\").astype(int)\n    return df\n"

# 1) Librerías: solo se instalan las que falten
paquetes = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "seaborn": "seaborn",
            "imblearn": "imbalanced-learn", "joblib": "joblib"}
faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

# 2) Datos: NSL-KDD en la carpeta data/ (se descarga una sola vez)
BASE_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
os.makedirs("data", exist_ok=True)
for nombre in ["KDDTrain+.txt", "KDDTest+.txt"]:
    destino = os.path.join("data", nombre)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BASE_URL + nombre.replace("+", "%2B"), destino)
    print(f"{nombre}: {os.path.getsize(destino) / 1e6:.1f} MB")

import numpy, pandas, sklearn
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__)

# 3) utils.py de la sesión 1: se crea si no está junto al cuaderno
if not os.path.exists("utils.py"):
    with open("utils.py", "w", encoding="utf-8") as f:
        f.write(UTILS_BASE)
    print("utils.py creado")
else:
    print("utils.py encontrado")

## Parte A · Perfila y limpia NSL-KDD

In [ ]:
import numpy as np, pandas as pd
from utils import load_nsl_kdd
tr = load_nsl_kdd("train"); te = load_nsl_kdd("test")
print(tr.dtypes.value_counts())
print("nulos:", tr.isna().sum().sum())
print("duplicados train:", tr.duplicated().sum(),
      "test:", te.duplicated().sum())
print(tr[["protocol_type", "service", "flag"]].nunique())

# registros del test que también están en train
cols = [c for c in tr.columns
        if c not in ("label", "difficulty", "is_attack")]
comunes = pd.merge(tr[cols].drop_duplicates(),
                   te[cols].drop_duplicates(), how="inner")
print("filas comunes train/test:", len(comunes))

In [ ]:
# columnas constantes: no aportan información
constantes = [c for c in tr.columns if tr[c].nunique() == 1]
print("constantes:", constantes)

1. ¿Cuántos duplicados hay? ¿Los eliminarías? *Tu respuesta:*
2. Elimina las columnas constantes (no aportan información). *Tu respuesta:*
3. ¿Qué riesgo hay si hay filas idénticas en train y test? *Tu respuesta:*

## Parte B · De 23 tipos de ataque a 4 categorías

In [ ]:
ATTACK_MAP = {"normal": "normal",
  **{a: "dos" for a in ["back","land","neptune","pod","smurf",
      "teardrop","apache2","udpstorm","processtable","mailbomb"]},
  **{a: "probe" for a in ["satan","ipsweep","nmap","portsweep",
      "mscan","saint"]},
  **{a: "r2l" for a in ["guess_passwd","ftp_write","imap","phf",
      "multihop","warezmaster","warezclient","spy","xlock",
      "xsnoop","snmpguess","snmpgetattack","httptunnel",
      "sendmail"]},
  **{a: "u2r" for a in ["buffer_overflow","loadmodule","rootkit",
      "perl","sqlattack","xterm","ps"]}}
for d in (tr, te):
    d["category"] = d["label"].map(ATTACK_MAP)
print(tr["category"].value_counts(dropna=False))
print(te["category"].value_counts(dropna=False))

In [ ]:
# ¿Qué ataques del test no están en el mapa?
print(sorted(te.loc[te["category"].isna(), "label"].unique()))

Amplía `ATTACK_MAP` con los ataques que falten y guarda la lógica en `utils.py` como `add_category(df)` ejecutando la celda siguiente.

In [ ]:
%%writefile -a utils.py

ATTACK_MAP = {"normal": "normal",
  **{a: "dos" for a in ["back","land","neptune","pod","smurf",
      "teardrop","apache2","udpstorm","processtable","mailbomb"]},
  **{a: "probe" for a in ["satan","ipsweep","nmap","portsweep",
      "mscan","saint"]},
  **{a: "r2l" for a in ["guess_passwd","ftp_write","imap","phf",
      "multihop","warezmaster","warezclient","spy","xlock",
      "xsnoop","snmpguess","snmpgetattack","httptunnel",
      "sendmail"]},
  **{a: "u2r" for a in ["buffer_overflow","loadmodule","rootkit",
      "perl","sqlattack","xterm","ps"]}}


def add_category(df):
    """Añade la columna category: normal, dos, probe, r2l o u2r."""
    df["category"] = df["label"].map(ATTACK_MAP)
    return df

In [ ]:
import importlib, utils
importlib.reload(utils)
from utils import add_category
tr = add_category(tr); te = add_category(te)
print(tr["category"].value_counts(dropna=False))

## Parte C · Codifica las variables categóricas

In [ ]:
from sklearn.preprocessing import OneHotEncoder
cat_cols = ["protocol_type", "service", "flag"]
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(tr[cat_cols])          # fit SOLO con train
Xc_tr = enc.transform(tr[cat_cols])
Xc_te = enc.transform(te[cat_cols])
print(Xc_tr.shape, Xc_te.shape)
print(enc.categories_[1][:10])

# codificación por frecuencia
freq = tr["service"].value_counts(normalize=True)
tr["service_freq"] = tr["service"].map(freq)
te["service_freq"] = te["service"].map(freq).fillna(0)

1. ¿Por qué `handle_unknown="ignore"`? *Tu respuesta:*
2. Ventajas y riesgos de la codificación por frecuencia frente a one-hot, en tres líneas. *Tu respuesta:*

In [ ]:
# La trampa: target encoding de service (tasa media de ataque por servicio)
from sklearn.tree import DecisionTreeClassifier
media = tr["is_attack"].mean()
m_train = tr.groupby("service")["is_attack"].mean()                      # solo train: correcto
m_todo = pd.concat([tr, te]).groupby("service")["is_attack"].mean()      # todo el dataset: fuga
m_test = te.groupby("service")["is_attack"].mean()                       # etiquetas del test: fuga grave
casos = {"solo train": (m_train, m_train),
         "todo el dataset": (m_todo, m_todo),
         "test con sus etiquetas": (m_train, m_test)}
for nombre, (mapa_tr, mapa_te) in casos.items():
    a = tr["service"].map(mapa_tr).fillna(media).to_frame("service_te")
    b = te["service"].map(mapa_te).fillna(media).to_frame("service_te")
    m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(a, tr["is_attack"])
    print(f"{nombre:24s} accuracy test = {m.score(b, te['is_attack']):.3f}")

¿Cuál infla la métrica y por qué? ¿Por qué aquí «todo el dataset» apenas cambia el resultado y sigue siendo un error? *Tu respuesta:*

## Parte D · Escala con lo que sabe el entrenamiento

In [ ]:
from sklearn.preprocessing import StandardScaler
num = ["duration", "src_bytes", "dst_bytes", "count",
       "srv_count", "serror_rate", "rerror_rate", "same_srv_rate"]
print(tr[num].describe().T[["mean", "std", "min", "max"]])
sc = StandardScaler().fit(tr[num])
Z_tr = sc.transform(tr[num]); Z_te = sc.transform(te[num])
print(pd.DataFrame(Z_tr, columns=num).describe().T[["mean", "std"]].round(3))

# cola pesada: log1p
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
tr["src_bytes"].plot(kind="hist", bins=50, ax=ax[0], title="src_bytes")
np.log1p(tr["src_bytes"]).plot(kind="hist", bins=50, ax=ax[1], title="log1p(src_bytes)")
plt.show()

- Verifica que la media de `Z_tr` es 0 y la desviación 1.
- En test no serán exactamente 0 y 1: ¿por qué es lo correcto? *Tu respuesta:*

## Parte E · Del auth.log a las características
IP legítimas en 10.10.10.0/24 y 12 IP de fuerza bruta en 203.0.113.0/24 (rango de documentación, RFC 5737). Semilla fija: todos obtenemos el mismo fichero.

In [ ]:
import numpy as np, datetime as dt
rng = np.random.default_rng(42); start = dt.datetime(2026, 10, 1)
users = ["root","admin","carlos","ana","luis","deploy","test","oracle","ubuntu","postgres"]
rows = []
for i in range(40):                                   # 40 IP legítimas
    ip = f"10.10.10.{10+i}"
    for t in rng.uniform(0, 7*24*3600, rng.integers(5, 40)):
        rows.append((start + dt.timedelta(seconds=float(t)), bool(rng.random() > 0.08),
                     str(rng.choice(users[2:6])), ip))
for i in range(12):                                   # 12 IP de fuerza bruta
    ip = f"203.0.113.{20+i}"; t0 = rng.uniform(0, 7*24*3600 - 3600)
    for t in t0 + np.sort(rng.uniform(0, 1800, rng.integers(60, 400))):
        rows.append((start + dt.timedelta(seconds=float(t)), False, str(rng.choice(users)), ip))
rows.sort(key=lambda r: r[0])

In [ ]:
import re
with open("auth.log", "w") as f:
    for ts, ok, user, ip in rows:
        v = "Accepted" if ok else "Failed"
        f.write(f"{ts:%b %d %H:%M:%S} srv sshd[{rng.integers(1000, 9999)}]: {v} password "
                f"for {user} from {ip} port {rng.integers(1024, 65000)} ssh2\n")
print(len(rows), "líneas")
pat = re.compile(r"^(\w{3}\s+\d+ \d\d:\d\d:\d\d) \S+ sshd\[\d+\]: (Failed|Accepted) "
                 r"password for (\S+) from (\S+) port (\d+)")
reg = []
for linea in open("auth.log"):
    m = pat.match(linea)
    if m:
        ts = pd.to_datetime("2026 " + m.group(1), format="%Y %b %d %H:%M:%S")
        reg.append({"ts": ts, "ok": m.group(2) == "Accepted", "user": m.group(3), "ip": m.group(4)})
log = pd.DataFrame(reg)
log.head()

In [ ]:
log["win"] = log["ts"].dt.floor("10min")
feat = (log.groupby(["ip", "win"])
          .agg(n=("ok", "size"), n_ok=("ok", "sum"),
               n_users=("user", "nunique"))
          .reset_index())
feat["n_fail"] = feat["n"] - feat["n_ok"]
feat["fail_ratio"] = feat["n_fail"] / feat["n"]
feat["hora"] = feat["win"].dt.hour
feat["hora_sin"] = np.sin(2*np.pi*feat["hora"]/24)
feat["hora_cos"] = np.cos(2*np.pi*feat["hora"]/24)
# etiqueta de laboratorio
feat["label"] = (feat["ip"].str.startswith("203.0.113.")
                 .astype(int))
print(feat.label.value_counts()); feat.head()

1. ¿Cuántas ventanas son de ataque? ¿Es una tasa base realista? *Tu respuesta:*
2. ¿Por qué no usamos `ip` como característica del modelo? *Tu respuesta:*
3. Añade tu característica: **ráfaga máxima**, intentos en el minuto más activo de cada ventana (pista: `log["min"] = log["ts"].dt.floor("1min")`).

In [ ]:
# Tu código para la ráfaga máxima

## Parte F · Tres divisiones y un pipeline sin fuga

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
X = feat[["n","n_ok","n_fail","n_users","fail_ratio","hora_sin","hora_cos"]]; y = feat["label"]

# 1) aleatoria estratificada
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
# 2) temporal: el 70 % de ventanas más antiguas para train
cut = feat["win"].quantile(0.7); tr_t = feat["win"] <= cut
# 3) por grupos (IP)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
i_tr, i_te = next(gss.split(X, y, groups=feat["ip"]))

print("IP compartidas, aleatoria:", len(set(feat.loc[Xa.index, "ip"]) & set(feat.loc[Xb.index, "ip"])))
print("IP compartidas, por grupos:", len(set(feat.loc[i_tr, "ip"]) & set(feat.loc[i_te, "ip"])))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
cols_num = ["duration","src_bytes","dst_bytes","count",
  "srv_count","serror_rate","rerror_rate","same_srv_rate"]
cols_cat = ["protocol_type","flag"]
num_pipe = Pipeline([("log", FunctionTransformer(np.log1p)),
                     ("sc", StandardScaler())])
prep = ColumnTransformer([("num", num_pipe, cols_num),
  ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat)])
pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(
  max_iter=1000, class_weight="balanced"))])
pipe.fit(tr[cols_num+cols_cat], tr["is_attack"])

X_te = te[cols_num+cols_cat]
pred = pipe.predict(X_te)
print(classification_report(te["is_attack"], pred))
joblib.dump(pipe, "pipe_ia2.joblib")

¿Qué clase se detecta peor? ¿Cuál de las tres divisiones es la más honesta para un detector que se usará en producción? *Tu respuesta:*

*Nota:* `log1p` solo es válido con valores no negativos, como en NSL-KDD.

## Parte G · Ampliación autónoma: desbalanceo extremo, u2r frente al resto

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score
from imblearn.over_sampling import SMOTE
for d in (tr, te): d["es_u2r"] = (d["category"] == "u2r").astype(int)
cols = cols_num + ["logged_in","num_root","num_file_creations","num_shells","root_shell"]
def evalua(modelo, Xtr, ytr, nombre):
    p = modelo.fit(Xtr, ytr).predict(te[cols])
    print(f"{nombre:16s} recall={recall_score(te.es_u2r, p):.3f} "
          f"precision={precision_score(te.es_u2r, p, zero_division=0):.3f}")
rf = lambda **kw: RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1, **kw)
evalua(rf(), tr[cols], tr.es_u2r, "sin tratar")
evalua(rf(class_weight="balanced"), tr[cols], tr.es_u2r, "class_weight")
Xs, ys = SMOTE(random_state=0, k_neighbors=3).fit_resample(tr[cols], tr.es_u2r)  # solo train
evalua(rf(), Xs, ys, "SMOTE en train")

¿Mejora el recall? ¿A qué coste en precisión? ¿Por qué aplicar SMOTE antes de dividir inflaría el resultado? *Tu respuesta:*

## Tabla de preprocesado (mínimo 12 columnas)
| Variable | Tipo | Tratamiento | Justificación |
|---|---|---|---|
| src_bytes | Numérica | log1p + z-score | Cola pesada |
| protocol_type | Categórica | One-hot | 3 valores sin orden |
| num_outbound_cmds | Constante | Eliminar | Sin información |
| | | | |

## Antes de entregar
- **Colab:** *Entorno de ejecución › Reiniciar y ejecutar todo* y *Archivo › Descargar › Descargar .ipynb*. Descarga también `utils.py`, `auth.log` y `pipe_ia2.joblib` desde el panel de archivos.
- **VS Code / Codespaces:** *Restart* y *Run All*; guarda y haz commit.
- Criterio de logro: ninguna transformación se ajusta con datos de test; las tres divisiones están implementadas y razonadas; la tabla tiene al menos 12 columnas.